In [1]:
# Change the branch to yours
# And change choose L4 GPU for now in Runtime > Change runtime type
BRANCH = "test-khalit"
REPO_URL = "git@github.com:St1p42/sglang.git"
REPO_DIR = "/content/sglang"

In [2]:
# Generate a public key for SSH - add the printed result to your GitHub account
%%bash
set -e

mkdir -p /root/.ssh
chmod 700 /root/.ssh

if [ ! -f /root/.ssh/id_ed25519 ]; then
  ssh-keygen -t ed25519 -C "colab" -f /root/.ssh/id_ed25519 -N ""
fi

ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null
chmod 600 /root/.ssh/known_hosts

echo "Public key:"
cat /root/.ssh/id_ed25519.pub

Generating public/private ed25519 key pair.
Your identification has been saved in /root/.ssh/id_ed25519
Your public key has been saved in /root/.ssh/id_ed25519.pub
The key fingerprint is:
SHA256:xJ6TQmv7aLHcg9BaOHraVe+I/jda6oAdDea/BgRdg9g colab
The key's randomart image is:
+--[ED25519 256]--+
|     + oo        |
|    o E. .       |
|     .+ o        |
|     +.* o       |
|     +* S        |
|    ++=* o       |
|   ..*=*. o      |
|  ..o.==+*o      |
|  .o.o++O+..     |
+----[SHA256]-----+
Public key:
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIHfcxFHVXC228JZfDUQoUpUOqcBxai2yQK+F5ivKKDKN colab


In [3]:
# Ssh test
%%bash
set -e
ssh -T git@github.com || true

Hi St1p42! You've successfully authenticated, but GitHub does not provide shell access.


In [4]:
import os
os.environ["BRANCH"] = BRANCH
os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = REPO_DIR

In [5]:
# Clone and checkout the branch indicated on the top
# After running, you should be able to see the repo in the Files in the left menu bar
%%bash
set -e

if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
git fetch origin
git checkout "$BRANCH"
git pull origin "$BRANCH"
git branch --show-current
git log --oneline -1

Branch 'test-khalit' set up to track remote branch 'test-khalit' from 'origin'.
Already up to date.
test-khalit
7b795d309 Fix how radix_cache_custom.py deals with partial node matches.


Cloning into '/content/sglang'...
Switched to a new branch 'test-khalit'
From github.com:St1p42/sglang
 * branch                test-khalit -> FETCH_HEAD


In [6]:
# See GPU/CPU/RAM info
%%bash
echo "=== GPU Info ==="
nvidia-smi || echo "No GPU attached or nvidia-smi failed"

echo -e "\n=== CPU Info ==="
lscpu | grep 'Model name\|Socket(s)\|Core(s) per socket\|Thread(s) per core'

echo -e "\n=== RAM Info ==="
free -h

=== GPU Info ===
Thu Apr 30 15:41:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

In [7]:
# Install
%%bash
set -e
cd /content/sglang

python -m pip uninstall -y sglang sgl-kernel flashinfer-python || true
python -m pip install -e python

Obtaining file:///content/sglang/python
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 2.6 MB/s eta 0:00:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.


In [8]:
# You should see something like:
# torch: 2.9.1+cu128
# sglang version: 0.5.6
# sglang path: ['/content/sglang/python/sglang']
# sglang spec origin: /content/sglang/python/sglang/__init__.py
# sgl_kernel: 0.3.18.post2
# cuda: True NVIDIA L4
import sys

# Make sure Python imports the actual package in your repo
sys.path = [p for p in sys.path if p != "/content/sglang"]
sys.path.insert(0, "/content/sglang/python")

import sglang, torch, sgl_kernel
from importlib.metadata import version

print("torch:", torch.__version__)
print("sglang version:", version("sglang"))
print("sglang path:", list(sglang.__path__))
print("sglang spec origin:", sglang.__spec__.origin)
print("sgl_kernel:", getattr(sgl_kernel, "__version__", "ok"))
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

torch: 2.9.1+cu128
sglang version: 0.5.6
sglang path: ['/content/sglang/python/sglang']
sglang spec origin: /content/sglang/python/sglang/__init__.py
sgl_kernel: 0.3.18.post2
cuda: True NVIDIA L4


In [ ]:
# Stop running the server when needed
!pkill -f "sglang.launch_server" || true

^C


In [ ]:
# Verify it's gone
!ps aux | grep "sglang.launch_server" | grep -v grep

In [ ]:
# Launch the server with vanilla radix-cache-impl flag
%%bash
cd /content/sglang
nohup python -m sglang.launch_server \
  --model-path Qwen/Qwen2.5-1.5B-Instruct \
  --port 30000 \
  > /content/sglang_server.log 2>&1 &

echo $! > /content/sglang_server.pid
sleep 20
tail -n 50 /content/sglang_server.log

2026-03-25 19:54:01.353630: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-25 19:54:01.372280: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774468441.394338   10727 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774468441.403434   10727 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774468441.420416   10727 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Launch the server with custom radix-cache-impl flag
%%bash
cd /content/sglang
nohup python -m sglang.launch_server \
  --model-path Qwen/Qwen2.5-1.5B-Instruct \
  --port 30000 \
  --radix-cache-impl custom \
  > /content/sglang_server.log 2>&1 &

echo $! > /content/sglang_server.pid
sleep 20
tail -n 50 /content/sglang_server.log

2026-03-27 06:05:45.947026: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-27 06:05:45.967149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774591545.990494   32981 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774591546.000121   32981 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774591546.018538   32981 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Check readiness
%%bash
curl -s http://127.0.0.1:30000/v1/models || true
echo
tail -n 50 /content/sglang_server.log

{"object":"list","data":[{"id":"Qwen/Qwen2.5-1.5B-Instruct","object":"model","created":1774591621,"owned_by":"sglang","root":"Qwen/Qwen2.5-1.5B-Instruct","parent":null,"max_model_len":32768}]}
[2026-03-27 06:06:18] Load weight end. type=Qwen2ForCausalLM, dtype=torch.bfloat16, avail mem=18.79 GB, mem usage=3.06 GB.
[2026-03-27 06:06:18] Using KV cache dtype: torch.bfloat16
[2026-03-27 06:06:18] KV Cache is allocated. #tokens: 570166, K size: 7.61 GB, V size: 7.61 GB
[2026-03-27 06:06:18] Memory pool end. avail mem=2.95 GB
[2026-03-27 06:06:18] Capture cuda graph begin. This can take up to several minutes. avail mem=2.32 GB
[2026-03-27 06:06:18] Capture cuda graph bs [1, 2, 4, 8, 12, 16, 24]
Capturing batches (bs=1 avail_mem=2.17 GB): 100%|██████████| 7/7 [00:01<00:00,  3.52it/s]
[2026-03-27 06:06:21] Capture cuda graph end. Time elapsed: 2.87 s. mem usage=0.15 GB. avail mem=2.17 GB.
[2026-03-27 06:06:21] max_total_num_tokens=570166, chunked_prefill_size=2048, max_prefill_tokens=16384, m

In [ ]:
# Sanity check of the server mode (for RADIX IMPL)
%%bash
grep "RADIX_CACHE_IMPL" /content/sglang_server.log

[2026-03-27 05:58:37] [RADIX_CACHE_IMPL] SELECTED custom source=main
[2026-03-27 05:58:37] [RADIX_CACHE_IMPL] CUSTOM RADIX_CACHE.PY source=main
[2026-03-27 05:58:38] [RADIX_CACHE_IMPL] SELECTED vanilla source=simulated
[2026-03-27 05:58:38] [RADIX_CACHE_IMPL] VANILLA RADIX_CACHE.PY source=simulated


In [9]:
# Patch so that we don't rely on the vLLM dependency just for the tokenizer
%%bash
set -e
cd /content/sglang/benchmark/multi_turn_chat

python - <<'PY'
from pathlib import Path

p = Path("bench_sglang.py")
s = p.read_text()

s = s.replace(
    "from vllm.transformers_utils.tokenizer import get_tokenizer",
    "from transformers import AutoTokenizer",
)
s = s.replace(
    "tokenizer = get_tokenizer(args.tokenizer, trust_remote_code=args.trust_remote_code)",
    "tokenizer = AutoTokenizer.from_pretrained(args.tokenizer, trust_remote_code=args.trust_remote_code)",
)

p.write_text(s)
print("patched bench_sglang.py")
PY

patched bench_sglang.py


In [ ]:
# Run multi-turn chat benchmark
%%bash
set -e
cd /content/sglang/benchmark/multi_turn_chat
python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct

Namespace(turns=4, num_qa=20, min_len_q=256, max_len_q=512, min_len_a=4, max_len_a=8, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=False, long=False, parallel=64, host='http://127.0.0.1', port=30000, backend='srt', device='auto', result_file='result.jsonl', raw_result_file=None)
Latency: 2.568


2026-03-27 05:24:38.717784: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-27 05:24:38.736671: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774589078.758914   21168 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774589078.766198   21168 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774589078.784489   21168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Check whether our custom methods get invoked
%%bash
LOG=/content/sglang_server.log

echo "=== CUSTOM RADIX TRACE COUNTS ==="
printf "match_prefix: "
grep -c "CUSTOM RADIX TRACE] match_prefix:start" "$LOG" || true

printf "cache_unfinished_req: "
grep -c "CUSTOM RADIX TRACE] cache_unfinished_req:start" "$LOG" || true

printf "cache_finished_req: "
grep -c "CUSTOM RADIX TRACE] cache_finished_req:start" "$LOG" || true

printf "evict: "
grep -c "CUSTOM RADIX TRACE] evict:" "$LOG" || true

printf "new-leaf: "
grep -c "mode=new-leaf" "$LOG" || true

printf "split: "
grep -c "mode=split" "$LOG" || true


=== CUSTOM RADIX TRACE COUNTS ===
match_prefix: 169
cache_unfinished_req: 88
cache_finished_req: 81
evict: 0
new-leaf: 168
split: 1


In [ ]:
# Run DSPy benchmark against the SGLang server
%%bash
set -e
cd /content/sglang

export DSP_CACHEBOOL=false

# Keep DSPy isolated so it does not overwrite the main SGLang environment.
# Colab's `python -m venv` is not always reliable, so install into a dedicated
# package directory and prepend it only for this benchmark process.
DSPY_SITE=/content/dspy-site
export DSPY_SITE
mkdir -p "$DSPY_SITE"
if ! python - <<'PY'
import importlib.util
import os
import sys
site_dir = os.environ['DSPY_SITE']
sys.path.insert(0, site_dir)
ok = importlib.util.find_spec('dspy') is not None
raise SystemExit(0 if ok else 1)
PY
then
  python -m pip install -q --target "$DSPY_SITE" dspy
fi

cd benchmark/dspy
PYTHONPATH="$DSPY_SITE${PYTHONPATH:+:$PYTHONPATH}" python bench_dspy_sglang.py \
  --url http://localhost \
  --port 30000 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --allow-rm-failure \
  --dev-size 50 \
  --num-threads 8

20 50
Question: At My Window was released by which American singer-songwriter?
Answer: John Townes Van Zandt
Question: What is the nationality of the chef and restaurateur featured in Restaurant: Impossible?
Answer: English
Relevant Wikipedia Titles: {'Restaurant: Impossible', 'Robert Irvine'}
For this dataset, training examples have input keys ['question'] and label keys ['answer']
For this dataset, dev examples have input keys ['question'] and label keys ['answer', 'gold_titles']
Question: What is the nationality of the chef and restaurateur featured in Restaurant: Impossible?
Predicted Answer: American




[2026-03-27T06:07:36.045908]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer` (str): often between 1 and 5 words
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your obj

In [ ]:
# Try with 1, 4, and 8 parallel conversations (with a total of 20)
# %%bash
# cd /content/sglang/benchmark/multi_turn_chat
# python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 1
# python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 4
# python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 8

Namespace(turns=4, num_qa=20, min_len_q=256, max_len_q=512, min_len_a=4, max_len_a=8, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=False, long=False, parallel=1, host='http://127.0.0.1', port=30000, backend='srt', device='auto', result_file='result.jsonl', raw_result_file=None)
Latency: 8.415
Namespace(turns=4, num_qa=20, min_len_q=256, max_len_q=512, min_len_a=4, max_len_a=8, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=False, long=False, parallel=4, host='http://127.0.0.1', port=30000, backend='srt', device='auto', result_file='result.jsonl', raw_result_file=None)
Latency: 3.387
Namespace(turns=4, num_qa=20, min_len_q=256, max_len_q=512, min_len_a=4, max_len_a=8, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=False, long=False, parallel=8, host='http://127.0.0.1', port=30000, backend='srt', device='auto', result_file='result.jsonl', raw_result_file=None)
Latency: 2.277


2026-03-18 05:38:50.320249: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 05:38:50.345216: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773812330.368630   83059 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773812330.376260   83059 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773812330.397856   83059 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 